# Preprocessing and Feature Engineering

## Clean the Kaggle data

Keep only the hybrid era (2014 onward) and the drivers on the 2026 grid (`data/recent/grid_2026.csv`). The output has one row per driver per Grand Prix, with a `podium` label (1 = finished in the top 3).

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
RECENT = Path("../data/recent")
PROCESSED = Path("../data/processed")
HYBRID_START = 2014


def read_raw(name):
    # The Kaggle CSVs use \N for missing values
    return pd.read_csv(RAW / f"{name}.csv", na_values="\\N")


races = read_raw("races")
results = read_raw("results")
qualifying = read_raw("qualifying")
drivers = read_raw("drivers")
constructors = read_raw("constructors")
status = read_raw("status")
grid_2026 = pd.read_csv(RECENT / "grid_2026.csv")

### Match the 2026 grid to Kaggle driver IDs

Drivers are matched by full name, with accents removed (Hülkenberg, Pérez). Three-letter codes aren't unique across eras (Vergne also used VER), so they aren't used for matching.

In [2]:
def name_key(first, last):
    text = f"{first} {last}"
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode().lower().strip()


drivers["name_key"] = [name_key(f, l) for f, l in zip(drivers.forename, drivers.surname)]
grid_2026["name_key"] = [name_key(f, l) for f, l in zip(grid_2026.first_name, grid_2026.last_name)]

grid_matched = grid_2026.merge(drivers[["driverId", "name_key"]], on="name_key", how="left")
unmatched = grid_matched[grid_matched.driverId.isna()]
print(f"Matched {grid_matched.driverId.notna().sum()} of {len(grid_matched)} drivers")
print("No Kaggle history (debuted 2025 or later):", ", ".join(unmatched.first_name + " " + unmatched.last_name))

current_driver_ids = grid_matched.driverId.dropna().astype(int)

Matched 18 of 22 drivers
No Kaggle history (debuted 2025 or later): Gabriel Bortoleto, Andrea Kimi Antonelli, Arvid Lindblad, Isack Hadjar


### Filter and join

In [3]:
hybrid_races = races.loc[races.year >= HYBRID_START, ["raceId", "year", "round", "circuitId", "name", "date"]]
hybrid_races = hybrid_races.rename(columns={"name": "race_name"})

quali_pos = qualifying[["raceId", "driverId", "position"]].rename(columns={"position": "quali_position"})

df = (
    results[results.driverId.isin(current_driver_ids)]
    .merge(hybrid_races, on="raceId", how="inner")
    .merge(drivers[["driverId", "code", "forename", "surname"]], on="driverId", how="left")
    .merge(constructors[["constructorId", "name"]].rename(columns={"name": "constructor"}), on="constructorId", how="left")
    .merge(status, on="statusId", how="left")
    .merge(quali_pos, on=["raceId", "driverId"], how="left")
)

df["date"] = pd.to_datetime(df["date"])
df["podium"] = (df["positionOrder"] <= 3).astype(int)

df = df[[
    "raceId", "year", "round", "date", "race_name", "circuitId",
    "driverId", "code", "forename", "surname", "constructorId", "constructor",
    "quali_position", "grid", "position", "positionOrder", "points", "status", "podium",
]].sort_values(["date", "positionOrder"]).reset_index(drop=True)

df.head()

,raceId,year,round,date,race_name,circuitId,driverId,code,forename,surname,constructorId,constructor,quali_position,grid,position,positionOrder,points,status,podium
0,900,2014,1,2014-03-16,Australian Grand Prix,1,4,ALO,Fernando,Alonso,6,Ferrari,5.0,5,4.0,4,12.0,Finished,0
1,900,2014,1,2014-03-16,Australian Grand Prix,1,822,BOT,Valtteri,Bottas,3,Williams,10.0,15,5.0,5,10.0,Finished,0
2,900,2014,1,2014-03-16,Australian Grand Prix,1,807,HUL,Nico,Hülkenberg,10,Force India,7.0,7,6.0,6,8.0,Finished,0
3,900,2014,1,2014-03-16,Australian Grand Prix,1,815,PER,Sergio,Pérez,10,Force India,16.0,16,10.0,10,1.0,Finished,0
4,900,2014,1,2014-03-16,Australian Grand Prix,1,1,HAM,Lewis,Hamilton,131,Mercedes,1.0,1,NaN,19,0.0,Engine,0


### Quick checks

In [4]:
print(f"Rows: {len(df):,}  |  Races: {df.raceId.nunique()}  |  Seasons: {df.year.min()}-{df.year.max()}")
print(f"Podium rate: {df.podium.mean():.1%}")
print("Missing values:")
print(df.isna().sum()[lambda s: s > 0])

Rows: 2,514  |  Races: 228  |  Seasons: 2014-2024
Podium rate: 20.2%
Missing values:
quali_position      5
position          338
dtype: int64


In [5]:
df.groupby("code").agg(starts=("raceId", "count"), podiums=("podium", "sum"), first_season=("year", "min")) \
  .sort_values("podiums", ascending=False)

,starts,podiums,first_season
code,,,
HAM,227,148,2014
VER,209,112,2015
BOT,228,67,2014
LEC,149,43,2018
PER,226,36,2014
SAI,208,27,2015
NOR,128,26,2019
RUS,128,15,2019
ALO,187,11,2014


In [6]:
PROCESSED.mkdir(exist_ok=True)
df.to_csv(PROCESSED / "results_hybrid_2026_drivers.csv", index=False)